# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ART001-coder/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Refresh / Content Opportunity Scoring** — same lane as W02, now on the full warehouse
release instead of the starter CSV. Data source: `hf://datasets/FlyRank/internship-warehouse`,
build `flyrank_pseudonymized_warehouse_release_v20260703`. Iterating on `month=2026-03` (mid-panel);
`2026-06` (the `_sample` table) stays sealed as the future test month, never touched for label logic.

> ⚠️ **Run this in Colab with your `HF_TOKEN` secret set** — that's the only place it can actually
> reach the gated warehouse. Cells below are written to run top-to-bottom without edits, but a
> couple of markdown cells have a `[[FILL AFTER RUN: ...]]` placeholder where the exact number can
> only come from the live query — read it off the cell output above and drop it in before you commit.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis.** Two grains matter here, and I'm keeping them separate on purpose:

- The **table grain** for `fact_content_daily_performance` is one row per
  `report_date × client_hash_id × content_hash_id` — one content item, one client, one day.
  That's what I verify with a grain-probe query in section 3.
- The **decision grain** — the thing my lane actually scores — is one row per **content item**
  (`content_hash_id`), aggregated up from that daily grain over a chosen window. An editor
  refreshes one page at a time; the daily table is the raw material, not the unit I rank.

**Tables.** `dim_content` (content metadata, joined on `content_hash_id`) + `dim_clients`
(per-client history coverage, joined on `client_hash_id`) + `fact_content_daily_performance`
(the daily time series I aggregate). I'm not touching `fact_content_query_90d` this week — its
window is a fixed trailing 90 days that overlaps recent months in a way that needs its own
careful alignment check (per `skills/flyrank/flyrank-data/SKILL.md`), and this contract doesn't
need query-mix signals yet.

**Time window.** I iterate on the `month=2026-03` partition of the daily fact table — a single
mid-panel calendar month, per the instructions (never the `_sample` table, which is the sealed
final month, 2026-06). Within that one month I split into two halves (first half vs second half
of March) to build features and a proxy — never reaching into April or any month after March.

In [10]:
%pip -q install duckdb huggingface_hub

import os, getpass
import duckdb
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

# Token order: env var -> Colab Secret -> prompt (last resort).
# Never paste the token directly into a cell -- this repo is public.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_march':  f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# COUNT(*) over Parquet touches metadata, not data -- near-free, and confirms we're pointed
# at the right release before anything else runs.
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:18} {n:>12,} rows')

dim_clients                 104 rows
dim_content             519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily           78,835,655 rows
fact_daily_march      9,841,378 rows


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Label / proxy.** There's no logged "should refresh" outcome, so — same discipline as W02 — I
build an observed proxy entirely *inside* March: split the month into two halves (day 1-15 vs
16-31), and define `declined = 1` when second-half GSC impressions dropped 20%+ against the
first half. This never reaches outside the month I'm iterating on, so it's safe to develop here
without peeking at the sealed test month.

**Features (five, built in section 3).** All come from `fact_content_daily_performance` summed
or averaged *within* the March window, plus one from `dim_content` (content age). Each gets its
own "knowable at the decision moment because…" line below — that's the actual leakage discipline
this notebook is testing.

**Context (never a feature).** `content_hash_id`, `client_hash_id` — join/group keys only, the
scrambled codes carry no signal themselves. `report_date` — used to build windows, not fed to
any model directly.

**Excluded, deliberately, with why:**
- **GA4 metrics on rows where `ga4_data_available IS NOT TRUE`.** These aren't "zero
  engagement" — they're "not measured yet" (`docs/data-dictionary.md`'s three-valued-flag
  warning: the flag can be `TRUE`, `FALSE`, *or* `NULL`, and only `IS TRUE` / `IS NOT TRUE`
  filters both correctly). Averaging them in would quietly tell the model "no GA4 history" looks
  identical to "measured, zero engagement," which is a different fact entirely.
- **`fact_content_query_90d` columns.** Excluded this week specifically because its 90-day
  window overlaps recent months in a way I haven't aligned against my March window yet — using
  it without that check risks pulling in information from outside the window I claim to be
  scoring on.
- **The second-half-of-March impression total, as a feature.** This is exactly what my proxy
  label is built from — using it as an input would be leakage. I deliberately re-add it in
  section 3's trap to *show* why, then remove it again.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three required proofs, in order: the grain holds, the slice's size and date span, and how many
rows survive an honest `IS TRUE` availability filter. Then the five-feature frame, then the trap.

In [11]:
# ---- Proof 1: grain. One row really is report_date x client x content. ----
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_daily_march']}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows violating the stated grain (report_date x client x content): {len(grain_check)}")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the stated grain (report_date x client x content): 0


,report_date,client_hash_id,content_hash_id,c


**Grain result:** an empty table above means zero rows share a `(report_date, client_hash_id,
content_hash_id)` combination — the grain I stated in section 1 holds on the March partition. If
this ever comes back non-empty, the unit-of-analysis sentence above is wrong and needs rewriting
before anything downstream is trusted.

In [12]:
# ---- Proof 2: the lane's slice size and date span for March. ----
slice_summary = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        COUNT(DISTINCT client_hash_id)  AS n_clients
    FROM {TABLES['fact_daily_march']}
""").df()

slice_summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,min_date,max_date,n_content_items,n_clients
0,9841378,2026-03-01,2026-03-31,331437,55


**Slice size & span:** `[[FILL AFTER RUN: 9841378]]` rows, spanning `[[FILL AFTER RUN: 2026-03-01]]`
to `[[FILL AFTER RUN: 2026-03-31]]`, across `[[FILL AFTER RUN: 331437]]` distinct content
items and `[[FILL AFTER RUN: 55]]` clients. The date span should read as the full calendar
month (2026-03-01 through 2026-03-31) — if it's short on either end, some clients' `gsc_data_start`
falls inside March, which is exactly the unbalanced-panel limitation named in section 4.

In [13]:
# ---- Proof 3: availability. Filter with IS TRUE, show how many rows survive. ----
availability = con.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)     AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 1.0
            / COUNT(*)                                                  AS pct_available
    FROM {TABLES['fact_daily_march']}
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,0.042064


**Availability result:** of `[[FILL AFTER RUN: 9841378]]` March rows,
`[[FILL AFTER RUN: 413966.0]]` (`[[FILL AFTER RUN: 0.042064]]`) have
`ga4_data_available IS TRUE` and are safe to use for any GA4-based feature. The rest are
`FALSE` or `NULL` — per section 2, I treat both the same way (excluded from GA4 averages, not
zero-filled), which is why the query uses `IS TRUE` rather than `= TRUE` (a plain `= TRUE` drops
`NULL` rows silently instead of counting them as unavailable on purpose).

## Five features (max five), each with an "available when?" line

*Build a small feature frame for the lane from the same March month. One line per feature:
knowable at the decision moment because…*

Decision moment = end of the March window (2026-03-31). Every feature below only touches rows
with `report_date <= 2026-03-31`, so nothing here reaches past the moment I'd actually be
scoring a page.

In [14]:
# Schema discovery first: dim_content's exact column names for "created at" and
# fact_daily's exact GA4 session/engagement column names aren't hardcoded here --
# discovered live, so this cell can't silently drift from the real schema.
dim_content_cols = con.sql(f"SELECT * FROM {TABLES['dim_content']} LIMIT 0").df().columns.tolist()
fact_cols = con.sql(f"SELECT * FROM {TABLES['fact_daily_march']} LIMIT 0").df().columns.tolist()

created_col = next((c for c in dim_content_cols if 'created' in c.lower()), None)
ga4_session_col = next((c for c in fact_cols if c.lower() in ('ga4_sessions', 'sessions') or ('session' in c.lower() and 'ga4' in c.lower())), None)
ga4_engaged_col = next((c for c in fact_cols if 'engaged' in c.lower()), None)

print("dim_content columns:", dim_content_cols)
print()
print("fact_daily columns:", fact_cols)
print()
print("Using created_col =", created_col, "| ga4_session_col =", ga4_session_col, "| ga4_engaged_col =", ga4_engaged_col)

dim_content columns: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

fact_daily columns: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', '

In [15]:
features_sql = f"""
    WITH march AS (
        SELECT *
        FROM {TABLES['fact_daily_march']}
        WHERE report_date <= DATE '2026-03-31'
    )
    SELECT
        content_hash_id,
        ANY_VALUE(client_hash_id)                                              AS client_hash_id,
        SUM(gsc_impressions)                                                   AS impressions_mar,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)          AS avg_position_mar,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)     AS active_days_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN {ga4_engaged_col or ga4_session_col} ELSE 0 END) * 1.0
            / NULLIF(SUM(CASE WHEN ga4_data_available IS TRUE THEN {ga4_session_col} ELSE 0 END), 0)
                                                                                AS ga4_engagement_rate_mar
    FROM march
    GROUP BY 1
    HAVING impressions_mar > 0
"""
feature_frame = con.sql(features_sql).df()

# fifth feature: content age at the decision moment, from dim_content
content_meta = con.sql(f"""
    SELECT content_hash_id, {created_col} AS content_created_at
    FROM {TABLES['dim_content']}
""").df()

feature_frame = feature_frame.merge(content_meta, on='content_hash_id', how='left')
feature_frame['content_age_days_mar'] = (
    pd.Timestamp('2026-03-31') - pd.to_datetime(feature_frame['content_created_at'])
).dt.days

print(f"{len(feature_frame):,} content items with at least one March impression")
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

176,738 content items with at least one March impression


,content_hash_id,client_hash_id,impressions_mar,avg_position_mar,active_days_mar,ga4_engagement_rate_mar,content_created_at,content_age_days_mar
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,5.331238,29,NaN,2026-02-12,47
1,content_67741cce996cfafa,client_62f4a7e64f5e0096,46.0,5.942308,16,NaN,2026-02-12,47
2,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,899.0,5.908100,31,NaN,2026-02-12,47
3,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,6.419872,17,NaN,2026-02-12,47
4,content_65c50dfe9d87a585,client_62f4a7e64f5e0096,3108.0,6.969536,30,NaN,2026-02-12,47


**The five features, and why each is knowable at the decision moment (end of March):**

1. **`impressions_mar`** — total GSC impressions summed over March. Knowable because it's
   already-elapsed, already-logged search data through the end of the month I'm scoring on —
   nothing here reaches into April.
2. **`avg_position_mar`** — mean GSC ranking position across March (excluding `0`, which per
   the data dictionary means "no data," not rank zero). Knowable for the same reason: it's a
   completed month's worth of already-observed ranking data.
3. **`active_days_mar`** — count of distinct days in March the page received any impressions.
   Knowable because it's a simple count over data that's fully in the past by month-end.
4. **`ga4_engagement_rate_mar`** — engaged-session share, computed *only* over rows where
   `ga4_data_available IS TRUE` (per section 2's exclusion). Knowable because it's measured GA4
   history already logged for March; restricting to the availability flag stops "not tracked
   yet" from masquerading as "zero engagement."
5. **`content_age_days_mar`** — days between `content_created_at` and 2026-03-31. Knowable
   because publish date is a fixed historical fact, set once, long before any decision point
   that comes after it.

## The trap: add one label-derived column, watch the score jump, then remove it

Same lesson as `notebooks/02_your_first_readable_model.ipynb`'s `trend_pct` leak — performed
here on real warehouse data instead of the starter CSV.

**The proxy label**, built entirely inside March: `declined = 1` when a page's second-half-of-
March GSC impressions dropped 20%+ against its first half. I'll fit an honest score using the
five features above, then deliberately add the *raw quantity the label was thresholded from* as
a sixth feature and watch precision spike toward 1.0 — because at that point the model isn't
predicting decline, it's just reading the answer back.

In [16]:
# Build the within-March halves needed for the proxy label (kept separate from feature_frame
# on purpose -- this is label-construction, not a feature).
halves_sql = f"""
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impr_first_half,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS impr_second_half
    FROM {TABLES['fact_daily_march']}
    WHERE report_date <= DATE '2026-03-31'
    GROUP BY 1
    HAVING impr_first_half >= 10
"""
halves = con.sql(halves_sql).df()

halves['pct_change_within_march'] = (
    (halves['impr_second_half'] - halves['impr_first_half']) / halves['impr_first_half']
)
halves['declined'] = (halves['pct_change_within_march'] <= -0.2).astype(int)

data = feature_frame.merge(halves[['content_hash_id', 'pct_change_within_march', 'declined']],
                            on='content_hash_id', how='inner')
print(f"{len(data):,} content items with enough March-1st-half volume for a label")
print(f"base rate (declined): {data['declined'].mean():.3f}")

120,513 content items with enough March-1st-half volume for a label
base rate (declined): 0.298


In [17]:
from sklearn.tree import DecisionTreeClassifier, export_text

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

honest_features = ['impressions_mar', 'avg_position_mar', 'active_days_mar',
                    'ga4_engagement_rate_mar', 'content_age_days_mar']

X_honest = data[honest_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = data['declined'].values

honest_tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42)
honest_tree.fit(X_honest, y)
honest_score = honest_tree.predict_proba(X_honest)[:, 1]

k = min(50, len(data))
print(f"HONEST Precision@{k}: {precision_at_k(honest_score, y, k):.3f}")
print(export_text(honest_tree, feature_names=honest_features))

HONEST Precision@50: 0.800
|--- active_days_mar <= 16.50
|   |--- active_days_mar <= 12.50
|   |   |--- class: 1
|   |--- active_days_mar >  12.50
|   |   |--- class: 1
|--- active_days_mar >  16.50
|   |--- content_age_days_mar <= 34.50
|   |   |--- class: 0
|   |--- content_age_days_mar >  34.50
|   |   |--- class: 0



In [18]:
# ---- The trap: add ONE label-derived column on purpose ----
leaky_features = honest_features + ['pct_change_within_march']
X_leaky = data[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)

leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42)
leaky_tree.fit(X_leaky, y)
leaky_score = leaky_tree.predict_proba(X_leaky)[:, 1]

print(f"LEAKY Precision@{k}: {precision_at_k(leaky_score, y, k):.3f}  <- looks amazing, and it should worry you")
print(export_text(leaky_tree, feature_names=leaky_features))

LEAKY Precision@50: 1.000  <- looks amazing, and it should worry you
|--- pct_change_within_march <= -0.20
|   |--- class: 1
|--- pct_change_within_march >  -0.20
|   |--- class: 0



In [19]:
# ---- Delete the leak, keep the honest number ----
FINAL_FEATURES = honest_features  # pct_change_within_march stays out, permanently
print("Final feature set (leak removed):", FINAL_FEATURES)
print(f"Honest Precision@{k} this contract stands on: {precision_at_k(honest_score, y, k):.3f}")

Final feature set (leak removed): ['impressions_mar', 'avg_position_mar', 'active_days_mar', 'ga4_engagement_rate_mar', 'content_age_days_mar']
Honest Precision@50 this contract stands on: 0.800


**What happened:** the leaky tree's precision jumps toward 1.0 — `[[FILL AFTER RUN: 1.00]]` versus the honest `[[FILL AFTER RUN: 0.80]]` — because
`pct_change_within_march` *is* the exact quantity `declined` was thresholded from. The printed
tree above will show it splitting almost entirely on that one column: it isn't predicting decline,
it's reading the label back through a keyhole. That's the same trap as `trend_pct` in notebook 02,
just rebuilt myself on real warehouse data instead of copying the lesson. The honest number —
built only from `FINAL_FEATURES`, none of them derived from the label — is the one this contract
actually stands behind.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [20]:
# One named limitation, verified: the unbalanced panel. Not every client has a full
# March of history -- some start mid-month, some have none yet.
panel_check = con.sql(f"""
    SELECT
        c.client_hash_id,
        c.gsc_data_start,
        COUNT(f.report_date) AS march_rows
    FROM {TABLES['dim_clients']} c
    LEFT JOIN {TABLES['fact_daily_march']} f
      ON f.client_hash_id = c.client_hash_id
    GROUP BY 1, 2
    ORDER BY march_rows ASC
    LIMIT 10
""").df()

print(f"Clients with zero or partial March coverage (bottom 10 by row count):")
panel_check

Clients with zero or partial March coverage (bottom 10 by row count):


,client_hash_id,gsc_data_start,march_rows
0,client_123b42d7ca0e1690,NaT,0
1,client_861cdcccf8049915,2025-11-05,0
2,client_46703b915c4762e0,NaT,0
3,client_835f9123c933bc01,2025-11-05,0
4,client_c353557474475e51,2026-04-30,0
5,client_a22068e339bf95f5,2026-05-24,0
6,client_c7c2962f1c9c3089,2026-05-14,0
7,client_7de9989c909e91a5,2026-05-18,0
8,client_f63f09ff4e81aa58,NaT,0
9,client_1d09b519bdde7c7a,2025-11-05,0


In [21]:
import os, getpass

In [22]:
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

**Named limitation: the unbalanced panel.** `dim_clients.gsc_data_start` differs per client —
some clients' tracking starts after 2026-03-01, so they contribute zero or partial rows to this
month's slice, not because they had no traffic but because they weren't onboarded yet. A flat
calendar-month window like `month=2026-03` silently treats "not tracked yet" the same as
"no content," which biases any March-only feature toward FlyRank's longer-tenured clients. The
fix isn't in this notebook — it's picking per-client windows anchored to each `gsc_data_start`
rather than one global calendar month, which is exactly what a later, deeper pass (or the
`w03_feature_leakage_check.ipynb` sibling) should do instead of what I did here.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this in Colab
      with `HF_TOKEN` set; I could not execute this myself (no network path to Hugging Face from
      my environment) — check every `[[FILL AFTER RUN: ...]]` placeholder is replaced with the
      real number before committing**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Five plain-words contract answers (unit of analysis, tables, time window, label/proxy,
      one deliberate exclusion)
- [x] Exactly three verification queries with outputs (grain check, row count + date span,
      availability with `IS TRUE`)
- [x] Five-feature frame with an "available when?" line per feature
- [x] The deliberate-leak experiment shown, then removed, honest number kept
- [x] One named, verified limitation (the unbalanced panel)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.